# Stellar Spectral Synthesis with Jorg

This notebook demonstrates how to synthesize stellar spectra using Jorg, the Python/JAX port of Korg.jl.

**Main parameters:**
- `Teff`: Effective temperature in Kelvin
- `logg`: Surface gravity (log₁₀ in cgs units)
- `m_H`: Metallicity [M/H] relative to solar
- `wavelengths`: Tuple of (start, end) in Angstroms
- `wavelength_spacing`: Resolution in Angstroms
- `vmic`: Microturbulence velocity in km/s

In [ ]:
import sys
sys.path.insert(0, '/Users/jdli/Project/Korg.jl/jorg_v2/src')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Import Jorg synthesis functions

print("✅ Imports complete")

ModuleNotFoundError: No module named 'jorg'

## Example 1: Solar Spectrum

Synthesize a solar spectrum with standard parameters: Teff=5780K, logg=4.44, [M/H]=0.0

In [ ]:
print("Synthesizing solar spectrum...")
wavelengths_solar, flux_solar, continuum_solar = synth(
    Teff=5780,                     # Effective temperature (K) - Sun
    logg=4.44,                     # Surface gravity (log10 cgs) - Sun
    m_H=0.0,                       # Metallicity [M/H] - Solar
    wavelengths=(5000, 5200),      # Wavelength range (Angstroms)
    wavelength_spacing=0.01        # Resolution (Angstroms)
)

print(f"\n✅ Synthesis complete!")
print(f"  Wavelength range: {wavelengths_solar[0]:.2f} - {wavelengths_solar[-1]:.2f} Å")
print(f"  Number of points: {len(wavelengths_solar)}")
print(f"  Mean flux: {np.mean(flux_solar):.4e} erg/s/cm²")
print(f"  Mean continuum: {np.mean(continuum_solar):.4e} erg/s/cm²")

In [ ]:
# Plot solar spectrum
plt.figure(figsize=(14, 6))
plt.plot(wavelengths_solar, flux_solar, 'b-', linewidth=0.5, label='Flux', alpha=0.8)
plt.plot(wavelengths_solar, continuum_solar, 'r--', linewidth=1.5, label='Continuum')
plt.xlabel('Wavelength (Å)', fontsize=12)
plt.ylabel('Flux (erg/s/cm²)', fontsize=12)
plt.title('Solar Spectrum (Teff=5780K, logg=4.44, [M/H]=0.0)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Example 2: Cool Giant Star

Synthesize the H-alpha region for a cool, metal-poor giant star.

In [ ]:
print("Synthesizing cool giant spectrum...")
wl_giant, flux_giant, cont_giant = synth(
    Teff=4500,                     # Cooler temperature
    logg=2.0,                      # Lower gravity (giant star)
    m_H=-0.5,                      # Metal-poor
    wavelengths=(6550, 6575),      # H-alpha region
    wavelength_spacing=0.005       # Higher resolution for line details
)

print(f"\n✅ Synthesis complete!")
print(f"  Wavelength range: {wl_giant[0]:.2f} - {wl_giant[-1]:.2f} Å")
print(f"  Number of points: {len(wl_giant)}")

In [ ]:
# Plot normalized spectrum (better for seeing line profiles)
plt.figure(figsize=(14, 6))
plt.plot(wl_giant, flux_giant/cont_giant, 'k-', linewidth=0.8)
plt.xlabel('Wavelength (Å)', fontsize=12)
plt.ylabel('Normalized Flux', fontsize=12)
plt.title('Cool Giant (Teff=4500K, logg=2.0, [M/H]=-0.5) - H-alpha Region', fontsize=14)
plt.axvline(6562.8, color='r', linestyle='--', alpha=0.5, label='H-alpha center')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Example 3: Using synthesize() for More Control

The `synthesize()` function provides more detailed control and returns a dictionary with additional information.

In [ ]:
print("Synthesizing with custom parameters...")
result = synthesize(
    Teff=6000,
    logg=4.0,
    m_H=0.2,                       # Metal-rich star
    wavelengths=(4850, 4870),      # H-beta region
    vmic=1.5,                      # Microturbulence (km/s)
    wavelength_spacing=0.01
)

# Access individual components
print(f"\n✅ Synthesis complete!")
print(f"  Available keys: {list(result.keys())}")
print(f"  Wavelength range: {result['wavelengths'][0]:.2f} - {result['wavelengths'][-1]:.2f} Å")
print(f"  Number of wavelength points: {len(result['wavelengths'])}")
print(f"  Mean flux: {np.mean(result['flux']):.4e} erg/s/cm²")
print(f"  Mean continuum: {np.mean(result['continuum']):.4e} erg/s/cm²")

In [ ]:
# Plot H-beta region
plt.figure(figsize=(14, 6))
normalized_flux = result['flux'] / result['continuum']
plt.plot(result['wavelengths'], normalized_flux, 'b-', linewidth=0.8)
plt.xlabel('Wavelength (Å)', fontsize=12)
plt.ylabel('Normalized Flux', fontsize=12)
plt.title('Metal-Rich Star (Teff=6000K, logg=4.0, [M/H]=+0.2) - H-beta Region', fontsize=14)
plt.axvline(4861.3, color='r', linestyle='--', alpha=0.5, label='H-beta center')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Example 4: Comparing Different Stellar Types

Compare spectra of hot (A-type), solar (G-type), and cool (K-type) stars in the same wavelength region.

In [ ]:
print("Comparing different stellar types...")

# Define wavelength region (G-band, sensitive to temperature)
wl_range = (4300, 4400)
spacing = 0.01

# Hot star (A-type)
print("  Synthesizing A-type star (Teff=8500K)...")
wl_hot, flux_hot, cont_hot = synth(
    Teff=8500, logg=4.0, m_H=0.0,
    wavelengths=wl_range,
    wavelength_spacing=spacing
)

# Solar-type (G-type)
print("  Synthesizing G-type star (Teff=5780K)...")
wl_solar, flux_solar, cont_solar = synth(
    Teff=5780, logg=4.44, m_H=0.0,
    wavelengths=wl_range,
    wavelength_spacing=spacing
)

# Cool star (K-type)
print("  Synthesizing K-type star (Teff=4500K)...")
wl_cool, flux_cool, cont_cool = synth(
    Teff=4500, logg=4.5, m_H=0.0,
    wavelengths=wl_range,
    wavelength_spacing=spacing
)

print("\n✅ All syntheses complete!")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# A-type star
axes[0].plot(wl_hot, flux_hot/cont_hot, 'b-', linewidth=0.8)
axes[0].set_ylabel('Normalized Flux', fontsize=11)
axes[0].set_title('A-type Star (Teff=8500K) - Hot', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.3, 1.05)

# G-type star (Sun)
axes[1].plot(wl_solar, flux_solar/cont_solar, 'g-', linewidth=0.8)
axes[1].set_ylabel('Normalized Flux', fontsize=11)
axes[1].set_title('G-type Star (Teff=5780K) - Solar', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.3, 1.05)

# K-type star
axes[2].plot(wl_cool, flux_cool/cont_cool, 'r-', linewidth=0.8)
axes[2].set_xlabel('Wavelength (Å)', fontsize=11)
axes[2].set_ylabel('Normalized Flux', fontsize=11)
axes[2].set_title('K-type Star (Teff=4500K) - Cool', fontsize=12)
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0.3, 1.05)

plt.suptitle('Comparison of Stellar Spectra (G-band Region)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Example 5: Metallicity Effects

Explore how metallicity affects spectral line strength.

In [ ]:
print("Comparing metallicity effects...")

# Fixed stellar parameters, varying metallicity
Teff = 5500
logg = 4.0
wl_range = (5160, 5180)  # Mg I triplet region
spacing = 0.01

metallicities = [-1.0, -0.5, 0.0, +0.3]
spectra = {}

for mh in metallicities:
    print(f"  Synthesizing [M/H]={mh:+.1f}...")
    wl, flux, cont = synth(
        Teff=Teff, logg=logg, m_H=mh,
        wavelengths=wl_range,
        wavelength_spacing=spacing
    )
    spectra[mh] = {'wl': wl, 'flux': flux, 'cont': cont}

print("\n✅ All syntheses complete!")

In [ ]:
# Plot metallicity comparison
plt.figure(figsize=(14, 7))

colors = ['purple', 'blue', 'green', 'red']
for i, mh in enumerate(metallicities):
    data = spectra[mh]
    norm_flux = data['flux'] / data['cont']
    plt.plot(data['wl'], norm_flux, color=colors[i], linewidth=0.8, 
             label=f'[M/H]={mh:+.1f}', alpha=0.8)

plt.xlabel('Wavelength (Å)', fontsize=12)
plt.ylabel('Normalized Flux', fontsize=12)
plt.title(f'Metallicity Effects on Spectrum (Teff={Teff}K, logg={logg})', fontsize=14)
plt.legend(fontsize=11, loc='lower right')
plt.grid(True, alpha=0.3)
plt.ylim(0.4, 1.05)
plt.tight_layout()
plt.show()

print("\n📝 Note: Higher metallicity → deeper and more numerous absorption lines")

## Example 6: Save Synthetic Spectrum

Save a synthetic spectrum to a text file for further analysis.

In [ ]:
# Synthesize a spectrum to save
print("Generating spectrum to save...")
wl, flux, cont = synth(
    Teff=5780, logg=4.44, m_H=0.0,
    wavelengths=(5000, 5100),
    wavelength_spacing=0.01
)

# Save to file
output_file = '/Users/jdli/Project/Korg.jl/jorg/examples/synthetic_spectrum.txt'
header = f"""# Synthetic Stellar Spectrum
# Generated with Jorg (Python port of Korg.jl)
# Stellar Parameters:
#   Teff = 5780 K
#   logg = 4.44
#   [M/H] = 0.0
# Columns: wavelength(Å) flux(erg/s/cm²) continuum(erg/s/cm²) normalized_flux
"""

# Create data array
normalized = flux / cont
data = np.column_stack([wl, flux, cont, normalized])

# Save with header
np.savetxt(output_file, data, header=header.strip(), 
           fmt=['%.4f', '%.6e', '%.6e', '%.6f'],
           delimiter='\t')

print(f"\n✅ Spectrum saved to: {output_file}")
print(f"  Number of points: {len(wl)}")
print(f"  File size: {np.ceil(len(wl) * 50 / 1024):.0f} KB (approx)")

## Summary

This notebook demonstrated:

1. **Basic synthesis** with `synth()` for quick spectral generation
2. **Cool giant stars** with different atmospheric parameters
3. **Using `synthesize()`** for more detailed control and metadata
4. **Comparing stellar types** (A, G, K spectral classes)
5. **Metallicity effects** on line strengths
6. **Saving spectra** to files for further analysis

**Key Parameters:**
- `Teff`: Effective temperature (3000-10000 K typical)
- `logg`: Surface gravity (0-5 in cgs log units)
- `m_H`: Metallicity (-3 to +1 typical)
- `wavelengths`: (start, end) in Angstroms
- `wavelength_spacing`: Resolution (0.005-0.1 Å typical)
- `vmic`: Microturbulence (0.5-3 km/s typical)

**Physics Features:**
- 277-species chemical equilibrium
- MARCS model atmospheres (56 layers)
- Complete continuum opacity (H⁻, Thomson, Rayleigh)
- Line absorption with Voigt profiles
- Exact radiative transfer solution